# Module 6: Multi-Agent (~8 min)

A lean orchestrator delegates to a specialist. Same team structure as
NFL Next Gen Stats: the broadcast inference pipeline doesn't do play
classification — it routes to a specialist that owns that domain.

Here Dussault handles roster/game queries directly but
delegates podcast research to a specialist agent with its own tools
and system prompt. The **agent-as-tool** pattern makes this seamless.

In [ ]:
!pip install -q strands-agents strands-agents-tools

import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

from strands import Agent, tool
from model_provider import get_model
from patriots_data import PODCAST_EPISODES
from dussault_tools import lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff

## The Agent-as-Tool Pattern

Wrap a specialist agent inside a `@tool`-decorated function. The
orchestrator sees it as just another tool in its toolkit. When the
orchestrator calls it, a full agent spins up, does its work silently
(`callback_handler=None`), and returns the result as a string.

This gives you:
- **Isolation** — specialist has its own tools + system prompt
- **Silent execution** — no streaming output from the inner agent
- **Clean delegation** — orchestrator decides WHEN to route, specialist decides HOW

In [ ]:
# --- Podcast specialist tools ---

@tool
def search_podcast_episodes(query: str) -> str:
    """Search podcast episodes about the 2004 Patriots dynasty by keyword.

    Args:
        query: Search term (player name, topic, keyword)
    """
    query_lower = query.lower()
    matches = []
    for ep in PODCAST_EPISODES:
        searchable = f"{ep['title']} {ep['description']} {' '.join(ep.get('keywords', []))} {' '.join(ep.get('interviewees', []))}".lower()
        if query_lower in searchable:
            matches.append(ep)

    if not matches:
        return f"No podcast episodes found matching '{query}'. Try a player name or topic like 'trade', 'defense', 'Super Bowl'."

    results = []
    for ep in matches:
        interviewees = ", ".join(ep.get("interviewees", ["unknown"]))
        results.append(
            f"{ep['series']} Ep {ep['episode']}: \"{ep['title']}\" ({ep['duration_min']} min, {ep['date']})\n"
            f"  Guests: {interviewees}\n"
            f"  {ep['description'][:120]}..."
        )
    return f"{len(matches)} episode(s) found:\n\n" + "\n\n".join(results)


@tool
def get_episode_details(series: str, episode: str) -> str:
    """Get full details for a specific podcast episode.

    Args:
        series: Series name ("2004 Dynasty" or "Pats from the Past")
        episode: Episode number or identifier ("I", "II", "7", "48", etc.)
    """
    match = next(
        (ep for ep in PODCAST_EPISODES
         if ep["series"].lower() == series.lower() and str(ep["episode"]) == str(episode)),
        None
    )
    if not match:
        return f"Episode not found: {series} #{episode}. Available series: '2004 Dynasty' (I-IV), 'Pats from the Past' (7-54)."

    lines = [
        f"{match['series']} \u2014 Episode {match['episode']}: \"{match['title']}\"",
        f"Duration: {match['duration_min']} minutes",
        f"Published: {match['date']}",
        f"Interviewees: {', '.join(match.get('interviewees', ['not listed']))}",
        f"Description: {match['description']}",
        f"Keywords: {', '.join(match.get('keywords', []))}",
    ]
    return "\n".join(lines)

## Build the Specialist

The podcast research specialist is a full agent wrapped in `@tool`.
It has its own system prompt, its own tools (search + details), and
runs silently. The orchestrator never sees the specialist's internal
reasoning — just the final answer.

In [ ]:
@tool
def podcast_research_specialist(query: str) -> str:
    """Delegate podcast research to the specialist agent.
    Use this when the user asks about interviews, podcast episodes, or
    what players/coaches said about the 2004 season.

    Args:
        query: The research question about podcast content
    """
    specialist = Agent(
        model=get_model(),
        tools=[search_podcast_episodes, get_episode_details],
        system_prompt="""You are a podcast research specialist for the 2004 Patriots dynasty.
You search episode archives to find relevant interviews and content.
Always cite the specific episode, guest, and air date.
If multiple episodes are relevant, rank them by relevance to the query.""",
        callback_handler=None,
    )
    print(f"\n[DELEGATION] \U0001f3a7 Podcast specialist activated for: {query[:60]}")
    response = specialist(query)
    print(f"[DELEGATION] \u2705 Specialist responded")
    return str(response)

## Build the Orchestrator

The orchestrator has both the roster/game tools (handles directly)
AND the `podcast_research_specialist` tool (delegates). The model
decides which path based on the user's question.

In [ ]:
SYSTEM_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault with access to
both data tools AND a podcast research specialist.

You handle:
- Roster lookups, game results, player stats, coaching staff (directly)
- Questions about what players or coaches said, podcast interviews,
  behind-the-scenes stories (delegate to podcast_research_specialist)

When a user asks about interviews, quotes, what someone said about the
season, or podcast content, delegate to the podcast_research_specialist
tool with a clear description of what to find.

After getting the specialist's response, synthesize it with your own
knowledge to give a complete answer."""

orchestrator = Agent(
    model=get_model(),
    tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats,
           get_coaching_staff, podcast_research_specialist],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

print("\u2705 Orchestrator ready — routes directly or delegates to specialist")

## Test: Direct Query (Orchestrator Handles)

A stats question routes to the roster/game tools. The orchestrator
handles it without involving the specialist.

In [ ]:
result = orchestrator("What were Tom Brady's 2004 stats?")
print(result)

## Test: Podcast Delegation

A question about what players *said* routes to the podcast specialist.
Watch for the `[DELEGATION]` marker — that's the handoff happening.

In [ ]:
result = orchestrator("What did the podcasts say about how the Corey Dillon trade came together?")
print(result)

## What's Next

**Module 7** adds **evaluations** — LLM-as-judge scoring against the
Dussault standard plus trajectory evaluation to verify the agent
follows the lookup-before-claim workflow. Same quality bar as NGS
needing 90% directional approval from domain experts.